# ADTomo two-grid pipeline
Run the numbered scripts first, then inspect the same forward calculation here.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from adtomo import ForwardGrid, VelocityModel, predict_travel_times

data_dir = Path('data')
data = torch.load(data_dir / 'model_true.pt', weights_only=True)
model = VelocityModel(**data, trainable=False)
stations = pd.read_csv(data_dir / 'stations.csv')
events = pd.read_csv(data_dir / 'events.csv', dtype={'event_id': str})
picks = pd.read_csv(data_dir / 'picks.csv', dtype={'event_id': str, 'station_id': str})

In [ ]:
station = stations.iloc[0]
station_picks = picks[(picks.station_id == station.station_id) & (picks.phase_type == 'P')].copy()
events_by_id = events.set_index('event_id')
station_event_ids = pd.unique(station_picks.event_id)
station_events = events_by_id.loc[station_event_ids]
station_spherical = torch.tensor([station.longitude, station.latitude, station.depth_km], dtype=torch.float64)
events_spherical = torch.tensor(station_events[['longitude', 'latitude', 'depth_km']].values, dtype=torch.float64)
grid = ForwardGrid(station_spherical, events_spherical, model, spacing=5.0)
grid_event_indices = torch.tensor(pd.Index(station_event_ids).get_indexer(station_picks.event_id))
predict_travel_times(model, grid, 'P', event_indices=grid_event_indices)[:5]